# Cybercrime Evolution (2015–2024) – Data Loading and Initial EDA

This notebook starts the technical work for my MSc project on the evolution of cybercrime using FBI IC3 data.
The goals in this section are:
- Load the consolidated crime type panel (2015–2024).
- Run basic exploratory data analysis (EDA) to understand structure, coverage, and data quality.
- Prepare clean yearly summaries that can feed later trend analysis and visualisation.

The master dataset used here is the curated crime-type panel compiled from the annual IC3 reports
after manual validation and correction of obvious extraction errors (especially for 2016).


In [1]:
# Core libraries
import pandas as pd
import numpy as np

# Display settings for easier inspection
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.0f}".format)

In [26]:
# Edit this cell before running

MASTER_CSV  = 'data/processed/ic3_master_crime_panel_2015_2024.csv'

# Column names as they appear in the master CSV
COL_CRIME   = 'Crime Type'
COL_YEAR    = 'Year'
COL_VICTIMS = 'Victim Count'
COL_LOSS    = 'Loss $'

# (one row per crime, columns like 2015_Victim Count, 2016_Victim Count)
IS_LONG = True

# Only needed when IS_LONG = False
YEAR_RANGE      = range(2015, 2025)
VICTIMS_SUFFIX  = '_Victim Count'
LOSS_SUFFIX     = '_Loss $'

In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os

warnings.filterwarnings('ignore')

os.makedirs('data/processed', exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)

print("Libraries loaded")

Libraries loaded


In [28]:
master = pd.read_csv(MASTER_CSV)
master.columns = [c.strip() for c in master.columns]

print("Shape:", master.shape)
print("Columns:", list(master.columns))

master.head()

Shape: (41, 21)
Columns: ['Crime Type', '2015 Victim Count', '2016 Victim Count', '2017 Victim Count', '2018 Victim Count', '2019 Victim Count', '2020 Victim Count', '2021 Victim Count', '2022 Victim Count', '2023 Victim Count', '2024 Victim Count', '2015 Loss ($)', '2016 Loss ($)', '2017 Loss ($)', '2018 Loss ($)', '2019 Loss ($)', '2020 Loss ($)', '2021 Loss ($)', '2022 Loss ($)', '2023 Loss ($)', '2024 Loss ($)']


,Crime Type,2015 Victim Count,2016 Victim Count,2017 Victim Count,2018 Victim Count,2019 Victim Count,2020 Victim Count,2021 Victim Count,2022 Victim Count,2023 Victim Count,2024 Victim Count,2015 Loss ($),2016 Loss ($),2017 Loss ($),2018 Loss ($),2019 Loss ($),2020 Loss ($),2021 Loss ($),2022 Loss ($),2023 Loss ($),2024 Loss ($)
0,419/Overpayment,"30,855",25716,"23,135","15,512","15,395","10,988","6,108","6,183","4,144","2,705","49,217,119","56,004,836","53,450,830","53,225,507","55,820,212","51,039,922","33,407,671","38,335,772","27,955,195","21,452,521"
1,Advanced Fee,"16,445",15075,"16,368","16,362","14,607","13,020","11,034","11,264","8,045","7,097","50,721,226","59,139,152","57,861,324","92,271,682","100,602,297","83,215,405","98,694,137","104,325,444","134,516,577","102,074,512"
2,Auction,"21,510",0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"18,906,416",0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,BEC/EAC,"7,837",12005,"15,690","20,373","23,775","19,369","19,954","21,832","21,489","21,442","246,226,016","360,513,961","676,151,185","1,297,803,489","1,776,549,688","1,866,642,107","2,395,953,296","2,742,354,049","2,946,830,270","2,770,151,146"
4,Botnet,NaN,0,NaN,NaN,NaN,NaN,NaN,568,540,587,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"17,099,378","22,422,708","8,860,202"


In [29]:
# List all columns to see years and metrics available
master.columns.tolist()

['Crime Type',
 '2015 Victim Count',
 '2016 Victim Count',
 '2017 Victim Count',
 '2018 Victim Count',
 '2019 Victim Count',
 '2020 Victim Count',
 '2021 Victim Count',
 '2022 Victim Count',
 '2023 Victim Count',
 '2024 Victim Count',
 '2015 Loss ($)',
 '2016 Loss ($)',
 '2017 Loss ($)',
 '2018 Loss ($)',
 '2019 Loss ($)',
 '2020 Loss ($)',
 '2021 Loss ($)',
 '2022 Loss ($)',
 '2023 Loss ($)',
 '2024 Loss ($)']

In [30]:
# General information about data types and non-null counts
master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Crime Type         41 non-null     object 
 1   2015 Victim Count  35 non-null     float64
 2   2016 Victim Count  41 non-null     int64  
 3   2017 Victim Count  35 non-null     float64
 4   2018 Victim Count  32 non-null     float64
 5   2019 Victim Count  32 non-null     float64
 6   2020 Victim Count  32 non-null     float64
 7   2021 Victim Count  28 non-null     float64
 8   2022 Victim Count  23 non-null     float64
 9   2023 Victim Count  26 non-null     float64
 10  2024 Victim Count  26 non-null     float64
 11  2015 Loss ($)      35 non-null     float64
 12  2016 Loss ($)      35 non-null     float64
 13  2017 Loss ($)      35 non-null     float64
 14  2018 Loss ($)      32 non-null     float64
 15  2019 Loss ($)      32 non-null     float64
 16  2020 Loss ($)      32 non-nu

In [31]:
# Separate complaint (victim count) and loss columns for later analysis
complaint_cols = [c for c in master.columns if "Victim Count" in c]
loss_cols       = [c for c in master.columns if "Loss ($)" in c]

complaint_cols, loss_cols

(['2015 Victim Count',
  '2016 Victim Count',
  '2017 Victim Count',
  '2018 Victim Count',
  '2019 Victim Count',
  '2020 Victim Count',
  '2021 Victim Count',
  '2022 Victim Count',
  '2023 Victim Count',
  '2024 Victim Count'],
 ['2015 Loss ($)',
  '2016 Loss ($)',
  '2017 Loss ($)',
  '2018 Loss ($)',
  '2019 Loss ($)',
  '2020 Loss ($)',
  '2021 Loss ($)',
  '2022 Loss ($)',
  '2023 Loss ($)',
  '2024 Loss ($)'])

In [32]:
# Simple missing-value profile
missing = (
    master.isna()
          .sum()
          .reset_index()
          .rename(columns={"index": "column", 0: "n_missing"})
          .sort_values("n_missing", ascending=False)
)

missing

,column,n_missing
8,2022 Victim Count,18
19,2023 Loss ($),15
10,2024 Victim Count,15
9,2023 Victim Count,15
20,2024 Loss ($),15
18,2022 Loss ($),15
17,2021 Loss ($),13
7,2021 Victim Count,13
15,2019 Loss ($),9
4,2018 Victim Count,9


# Note:
# Missing values represent non-reported categories, not zero.
# These are handled by exclusion in later analysis.

In [33]:
# Summary statistics across crime types for each year (complaints)
complaint_summary = (
    master[complaint_cols]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)

complaint_summary

,column,count,mean,std,min,25%,50%,75%,max
0,2015 Victim Count,35,"10,536","13,059",62,"1,110","5,458","16,883","67,375"
1,2016 Victim Count,41,"8,980","14,057",0,295,"2,673","14,546","81,029"
2,2017 Victim Count,35,"10,130","15,594",0,421,"3,089","15,737","84,079"
3,2018 Victim Count,32,"13,242","16,647",77,"1,468","8,986","16,186","65,116"
4,2019 Victim Count,32,"15,660","26,712",39,"1,343","9,304","15,422","140,491"
5,2020 Victim Count,32,"24,388","50,495",52,"1,869","10,680","19,678","269,560"
6,2021 Victim Count,28,"27,686","64,648",395,"1,947","11,456","21,396","342,494"
7,2022 Victim Count,23,"29,815","61,227",568,"2,691","11,779","29,226","300,497"
8,2023 Victim Count,26,"26,604","57,997",540,"3,050","9,554","21,061","298,878"
9,2024 Victim Count,26,"25,072","40,689",441,"3,168","11,995","21,432","193,407"


In [34]:
# Summary statistics across crime types for each year (losses)
loss_summary = (
    master[loss_cols]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "column"})
)

loss_summary

,column,count,mean,std,min,25%,50%,75%,max
0,2015 Loss ($),35,"35,433,186","56,422,521",0,"1,474,484","13,126,123","42,490,514","246,226,016"
1,2016 Loss ($),35,"41,248,444","73,014,793",0,"1,315,490","12,278,714","52,096,414","360,513,961"
2,2017 Loss ($),35,"49,159,769","118,474,534",0,"358,600","12,569,185","56,719,290","676,151,185"
3,2018 Loss ($),32,"106,744,248","232,397,262","10,193","4,261,558","49,356,314","104,750,266","1,297,803,489"
4,2019 Loss ($),32,"137,003,334","319,014,521","49,589","6,252,244","51,020,305","121,150,027","1,776,549,688"
5,2020 Loss ($),32,"157,700,521","337,896,801",0,"6,388,241","61,712,667","158,476,546","1,866,642,107"
6,2021 Loss ($),28,"273,124,146","526,849,384","142,643","6,681,429","73,563,306","293,074,206","2,395,953,296"
7,2022 Loss ($),26,"420,407,697","807,932,024","577,464","35,348,871","111,006,116","368,142,134","3,311,742,206"
8,2023 Loss ($),26,"474,893,738","1,024,675,700","1,213,317","23,805,830","110,353,322","372,949,992","4,570,275,683"
9,2024 Loss ($),26,"618,785,069","1,368,388,930","519,424","14,717,997","158,386,278","395,432,018","6,570,639,864"


In [35]:
# Compute yearly totals of complaints and losses across all crime types
yearly_totals = []

for c in complaint_cols:
    year = int(c.split()[0])
    yearly_totals.append(
        {"Year": year,
         "Metric": "complaints",
         "Total": master[c].sum(skipna=True)}
    )

for c in loss_cols:
    year = int(c.split()[0])
    yearly_totals.append(
        {"Year": year,
         "Metric": "losses",
         "Total": master[c].sum(skipna=True)}
    )

yearly_totals_df = (
    pd.DataFrame(yearly_totals)
      .sort_values(["Year", "Metric"])
      .reset_index(drop=True)
)

yearly_totals_df

,Year,Metric,Total
0,2015,complaints,"368,762"
1,2015,losses,"1,240,161,527"
2,2016,complaints,"368,194"
3,2016,losses,"1,443,695,530"
4,2017,complaints,"354,567"
5,2017,losses,"1,720,591,932"
6,2018,complaints,"423,743"
7,2018,losses,"3,415,815,949"
8,2019,complaints,"501,119"
9,2019,losses,"4,384,106,683"


In [37]:
OUT = 'data/processed/ic3_panel_long_harmonised.csv'

panel.to_csv(OUT, index=False)

print("Saved:", OUT)
print("Shape:  ", panel.shape)
print("Columns:", list(panel.columns))

Saved: data/processed/ic3_panel_long_harmonised.csv
Shape:   (41, 21)
Columns: ['Crime Type Harmonised', '2015 Victim Count', '2016 Victim Count', '2017 Victim Count', '2018 Victim Count', '2019 Victim Count', '2020 Victim Count', '2021 Victim Count', '2022 Victim Count', '2023 Victim Count', '2024 Victim Count', '2015 Loss ($)', '2016 Loss ($)', '2017 Loss ($)', '2018 Loss ($)', '2019 Loss ($)', '2020 Loss ($)', '2021 Loss ($)', '2022 Loss ($)', '2023 Loss ($)', '2024 Loss ($)']
